In [1]:
import os
from circuit_tracer.subgraph.prune import prune_graph_pipeline
from circuit_tracer.subgraph.api import generate_graph, get_feature, save_subgraph
from circuit_tracer.subgraph.classify import classify_features
from circuit_tracer.subgraph.group import grouping_pipeline
from pathlib import Path
import torch

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils.create_graph_files import create_graph_files  
from circuit_tracer.graph import Graph, prune_graph, compute_graph_scores
from transformers import AutoModelForCausalLM, AutoTokenizer
import shap

## Load LLM

In [4]:
model_name = "google/gemma-2-2b" # google/gemma-scope-2-4b-it crosscoder/layer_9_17_22_29_width_65k_l0_medium
# transcoder_name = "mntss/clt-gemma-2-2b-2.5M"
# backend = 'transformerlens'  # change to 'nnsight' for the nnsight backend!
# model = ReplacementModel.from_pretrained(
#     model_name, transcoder_name, dtype=torch.bfloat16, backend=backend
# )

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
# set model decoder to true
model.config.is_decoder = True
# ensure task_specific_params is initialized (avoid NoneType assignment error)
if model.config.task_specific_params is None:
    model.config.task_specific_params = {}
# set text-generation params under task_specific_params
model.config.task_specific_params["text-generation"] = {
    "do_sample": False, # set to False for deterministic output
    "max_new_tokens": 1, # set to 1 for single-token generation
    "temperature": 0, # set to 0 for deterministic output
    # "no_repeat_ngram_siz e": 2, 
}

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

## Use SHAP for token weights

In [6]:
explainer = shap.Explainer(model, tokenizer)

In [7]:
prompts = ['Fact: The capital of the state containing Dallas is']

In [8]:
shap_values = explainer(prompts)

In [9]:
print(shap_values)

.values =
array([[[-3.50681455e-06],
        [ 2.76400235e+00],
        [-8.72354661e-01],
        [ 2.00342429e+00],
        [ 4.72185472e+00],
        [ 1.18334559e+00],
        [ 1.11659276e-01],
        [ 2.30677287e+00],
        [ 9.67523081e-01],
        [ 5.83311160e+00],
        [ 2.01544604e+00]]])

.base_values =
array([[-21.22565525]])

.data =
(array(['', 'Fact', ':', ' The', ' capital', ' of', ' the', ' state',
       ' containing', ' Dallas', ' is'], dtype=object),)


In [ ]:
from scipy.special import softmax
token_weights = softmax(shap_values.values.squeeze()) # How to normalize Shap values? softmax, relu first then normalize, ...
print(token_weights)

[0.00198786 0.03153391 0.00083086 0.01473883 0.22338926 0.00649094
 0.00222269 0.01996207 0.0052309  0.67869559 0.01491708]


## Generate graph if not exist

In [ ]:
from generate_new_graphs import download_graph
download_graph(
    prompt=prompts[0],
    slug="austin",
    out_path="temp_graph_files/austin.json",
)

## ASG Pipeline

### Prune

In [ ]:
json_path = "temp_graph_files/austin.json"
# json_path = "temp_graph_files/future-qwen.json"
source_set = 'clt-hp' #'clt-hp' # gemmascope-transcoder-16k
# token_weights = [0.00198786, 0.03153391, 0.00083086, 0.01473883, 0.22338926, 0.00649094,
#  0.00222269, 0.01996207, 0.0052309, 0.67869559, 0.01491708]
# token_weights = [0, 0, 0, 0, 1/3, 0, 0, 1/3, 0, 1/3, 0]
# token_weights = [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1/3,0,0,1/3,0,1/3,0,0,0,0,0,0,0,0,0,0,0]
kept_ids, pruned_adj, node_inf, node_rel, attr, metadata = prune_graph_pipeline(
    json_path=json_path,
    logit_weights='target',
    token_weights=token_weights,
    node_influence_threshold=0.8,
    edge_influence_threshold=0.9,
    node_relevance_threshold=0.7,
    edge_relevance_threshold=0.8,
    keep_all_tokens_and_logits=False,
)

In [5]:
print(len(kept_ids))

26


In [6]:
for i, node_id in enumerate(kept_ids):
    print(node_id, attr[node_id].get("clerp", ""), node_inf[i].item(), node_rel[i].item())

16_89970_9 Texas 0.5723963379859924 0.6424320340156555
18_45367_10 US cities 0.6881327033042908 0.5960058569908142
18_56027_10 Dallas 0.7275407910346985 0.6118998527526855
19_8271_10 Foreign words 0.7114723324775696 0.5478754043579102
19_62362_10 city names 0.7174118757247925 0.5785500407218933
20_44686_9 Texas locations/legal contexts 0.5682745575904846 0.5120062232017517
20_14775_10 XML code 0.6804824471473694 0.6067379117012024
20_44686_10 Texas locations/legal contexts 0.5273332595825195 0.5191220045089722
20_74108_10 capital 0.5873730778694153 0.5618082284927368
21_16875_10 cities 0.6174425482749939 0.6015009880065918
21_61721_10 code/documentation 0.6148593425750732 0.5810048580169678
21_84338_10 Cities and locations 0.5839060544967651 0.5022703409194946
22_11998_10 Texas and Dallas 0.5599021315574646 0.494268000125885
22_26199_10 Texas 0.7579998970031738 0.5372506976127625
22_32893_10 Texas legal documents 0.5907537937164307 0.522559642791748
22_43081_10 technical contexts 0.668

In [7]:
print(attr['1_89326_9'])

{'1_89326_9': {'node_id': '1_89326_9', 'feature': 3989790454, 'layer': '1', 'ctx_idx': 9, 'feature_type': 'cross layer transcoder', 'token_prob': 0, 'is_target_logit': False, 'run_idx': 0, 'reverse_ctx_idx': 0, 'jsNodeId': '1_89326-0', 'clerp': 'Dallas', 'influence': 0.4841025769710541, 'activation': 9.1875}, '7_24743_9': {'node_id': '7_24743_9', 'feature': 306318368, 'layer': '7', 'ctx_idx': 9, 'feature_type': 'cross layer transcoder', 'token_prob': 0, 'is_target_logit': False, 'run_idx': 0, 'reverse_ctx_idx': 0, 'jsNodeId': '7_24743-0', 'clerp': 'Texas legal matters', 'influence': 0.4604751169681549, 'activation': 3.515625}, '8_21080_9': {'node_id': '8_21080_9', 'feature': 222383496, 'layer': '8', 'ctx_idx': 9, 'feature_type': 'cross layer transcoder', 'token_prob': 0, 'is_target_logit': False, 'run_idx': 0, 'reverse_ctx_idx': 0, 'jsNodeId': '8_21080-0', 'clerp': 'Texas legal contexts', 'influence': 0.5893152952194214, 'activation': 1.546875}, '16_89970_9': {'node_id': '16_89970_9', 

### Classify features

Heuristic classify alg is terrible, needs improvement or just use LLM

In [ ]:
# feature_types = classify_features(kept_ids, attr, metadata)
# print(feature_types)

IndexError: list index out of range

In [7]:
node_ids = [node_id for node_id in kept_ids if attr[node_id].get("feature_type", "") == "cross layer transcoder"]
# remove 21_61721_10, 22_74186_10
node_ids = [node_id for node_id in node_ids if node_id not in ["21_61721_10", "22_74186_10"]]
print(node_ids)

['16_89970_9', '18_45367_10', '18_56027_10', '19_8271_10', '19_62362_10', '20_44686_9', '20_14775_10', '20_44686_10', '20_74108_10', '21_16875_10', '21_84338_10', '22_11998_10', '22_26199_10', '22_32893_10', '22_43081_10', '22_81571_10', '23_29602_10', '23_31673_10', '23_59746_10', '24_79427_10']


In [8]:
labels = {node_id: attr[node_id].get("clerp", "") for node_id in node_ids }
print(labels)

{'16_89970_9': 'Texas', '18_45367_10': 'US cities', '18_56027_10': 'Dallas', '19_8271_10': 'Foreign words', '19_62362_10': 'city names', '20_44686_9': 'Texas locations/legal contexts', '20_14775_10': 'XML code', '20_44686_10': 'Texas locations/legal contexts', '20_74108_10': 'capital', '21_16875_10': 'cities', '21_84338_10': 'Cities and locations', '22_11998_10': 'Texas and Dallas', '22_26199_10': 'Texas', '22_32893_10': 'Texas legal documents', '22_43081_10': 'technical contexts', '22_81571_10': 'Texas locations and organizations', '23_29602_10': 'City or region names', '23_31673_10': 'Court appeals at specific location', '23_59746_10': 'Technical/Scientific writing', '24_79427_10': ' Kara'}


In [9]:
feature_types = {
    node_id: "Abstract" for node_id in node_ids
}
feature_types['1_89326_9'] = "Input"
print(feature_types)

{'16_89970_9': 'Abstract', '18_45367_10': 'Abstract', '18_56027_10': 'Abstract', '19_8271_10': 'Abstract', '19_62362_10': 'Abstract', '20_44686_9': 'Abstract', '20_14775_10': 'Abstract', '20_44686_10': 'Abstract', '20_74108_10': 'Abstract', '21_16875_10': 'Abstract', '21_84338_10': 'Abstract', '22_11998_10': 'Abstract', '22_26199_10': 'Abstract', '22_32893_10': 'Abstract', '22_43081_10': 'Abstract', '22_81571_10': 'Abstract', '23_29602_10': 'Abstract', '23_31673_10': 'Abstract', '23_59746_10': 'Abstract', '24_79427_10': 'Abstract', '1_89326_9': 'Input'}


In [12]:
supernodes = grouping_pipeline(
    node_ids = node_ids,
    labels = labels,
    feature_types = feature_types,
    prompt = prompts[0],
    model_name = 'gemini-2.5-flash',
)
print(supernodes)

[['Texas State Context', '16_89970_9', '22_11998_10', '22_26199_10'], ['General Geographic Entities', '18_45367_10', '19_62362_10', '21_16875_10', '21_84338_10', '23_29602_10'], ['Texas Legal/Specific Contexts', '20_44686_9', '20_44686_10', '22_32893_10', '22_81571_10', '23_31673_10'], ['Miscellaneous/Irrelevant', '19_8271_10', '20_14775_10', '22_43081_10', '23_59746_10', '24_79427_10']]


# TODO
List of things that need more experiments/improvements in descending order of importance for each step
## Pruning
1. Choose threshold (currently just sweep the threshold until we have ~30 nodes)
2. Initialize with Shap (normalize) (softmax | relu + normalize | +base_value then normalize)
3. Normalize adjacency matrix (currently adjacency.abs() then normalize (negative edges still contribute to influence and relevance))

## Grouping

Objectives to group features:
- similar embeddings (in classic SAE usually decoder vectors)
- simliar edges/logit effects (promoting the same feature or the same logit)
- similar contexts (activating on similar tokens, concepts, contexts)
- based on the claim we make about the mechanism (we might want to preserve the paths)

Approaches tried:
- Greedy constraint grouping
    + built distance graph based on BERT embeddings of autointerp (+ feature type) and group with antichain constraint (don't group nodes with path to each other)
    + input: autointerp + adj matrix
    + problems: constraints too strict, sensitive on the edges of the graph; did not consider local role of the features.
- Prompt LLM: 
    + Classify feature type based on feature details
    + Group using feature types + autointerp
    + problems: cover context and global role of the features, but not local role (in this exact graph) -> might not be mechanisticly sensible
    
Approaches to try:
- Clustering by (decoder vectors or autointerp) + edge weights similarity, constraint by difference in layers (may be sweep diff_layer = 0, 1, ... and merge iteratively)
- Higher order graphs




## Eval
1. Test intervention api
2. Find dataset
3. Hypothesis testing: Mechanism Preservation, Mechanism Localization, Minimality